In [ ]:
import os
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "US_Accidents_March23.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "sobhanmoosavi/us-accidents",
  file_path,
  pandas_kwargs={"dtype": {
      **{c: "category" for c in ["Source", "State", "Country", "Timezone",
          "Airport_Code", "Weather_Condition", "Wind_Direction", "Sunrise_Sunset",
          "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]},
      **{c: "float32" for c in ["End_Lat", "End_Lng", "Distance(mi)",
          "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
          "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]},
      "Severity": "int8",
  }},
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("5 pierwszych rekordów:", df.head())

# Analiza wypadków drogowych w USA

# Import wymaganych bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as mn
import plotly.graph_objects as go

# Przygotowanie danych

### Informacje

In [ ]:
df.info()

#### Kolumny ramki danych

In [ ]:
df.columns

#### Statystyki opisowe każdej kolumny numerycznej

In [ ]:
df.describe().T

**Obserwacje.** Zbiór zawiera 7 728 394 rekordy i 46 atrybutów. Rozkład `Distance(mi)` jest silnie prawostronnie skośny: mediana wynosi 0,03 mili, a średnia około 0,56 mili. Maksima temperatury 207°F, prędkości wiatru 1087 mph oraz ciśnienia 58,63 in wymagają kontroli wiarygodności; sama obecność ekstremum nie uzasadnia jednak usuwania wszystkich wartości odstających.

#### Kolumny numeryczne do analizy

In [ ]:
# Dane typu int, float i boolean
print(df.count(numeric_only=True))
print("Łączna liczba kolumn numerycznych:", len(df.count(numeric_only=True)))

### Procent brakujących wartości

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percentage = missing_values[missing_values!=0]/len(df)*100
print(" Procent brakujących wartości \n", missing_percentage)

**Wnioski o brakach.** Największe braki dotyczą `End_Lat` i `End_Lng` (po 44,03%), opadów (28,51%) i temperatury odczuwalnej (25,87%). Brak pomiaru opadów nie oznacza braku opadów. Usuwanie wszystkich niekompletnych rekordów mogłoby silnie zmniejszyć zbiór; wpływ zastosowanego czyszczenia na klasy sprawdzamy w dalszej części EDA.

Uzyskanie listy kolumn zawierających wartości puste

In [ ]:
null_cols = [i for i in df.columns if df[i].isnull().any()]
print(null_cols)

Sprawdzenie wzorca brakujących wartości w ramce danych

In [ ]:
mn.matrix(df[null_cols])

Usunięcie kolumn [End_Lng, End_Lat] zawierających dużo brakujących wartości oraz wierszy z brakami w wybranych kolumnach

In [ ]:
new_data_a = df.drop(columns=["End_Lng", "End_Lat"], axis=0)
new_data_b = new_data_a.dropna(subset = ['Visibility(mi)','Weather_Condition','Humidity(%)',
                                         'Temperature(F)','Wind_Direction','Pressure(in)',
                                         'Weather_Timestamp','Airport_Code','Timezone',
                                         'Zipcode','Sunrise_Sunset','Civil_Twilight',
                                         'Nautical_Twilight','Astronomical_Twilight',
                                         'City','Description'])

dataf = new_data_b.drop(columns = 'ID', axis=0)

### Usuwamy wiersze z brakującymi wartościami w wybranych kolumnach o niewielkim odsetku braków
### Dodatkowo braki w pozostałych pomiarach uzupełniamy średnią (tymczasowo)

In [ ]:
dataf["Wind_Chill(F)"] = dataf['Wind_Chill(F)'].fillna(dataf['Wind_Chill(F)'].mean())
dataf["Precipitation(in)"] = dataf['Precipitation(in)'].fillna(dataf['Precipitation(in)'].mean())
dataf["Wind_Speed(mph)"] = dataf['Wind_Speed(mph)'].fillna(dataf['Wind_Speed(mph)'].mean())

In [ ]:
final_data = dataf.reindex(copy=True)
final_data.head()

# Analiza wypadków według lokalizacji

In [ ]:
print("Kody stanów: ", final_data.State.unique())
print("Łączna liczba stanów w zbiorze danych: ", len(final_data.State.unique()))

In [ ]:
fig, ax = plt.subplots(figsize = (20,5))
c = sns.countplot(x="State", data=final_data, orient = 'v', palette = "crest_r", order = final_data['State'].value_counts().index)
c.set_title("Liczba wypadków według stanu");

In [ ]:
print("Nazwy miast: ", final_data.City.unique())
print("Łączna liczba miast w zbiorze danych: ", len(final_data.City.unique()))

In [ ]:
city_totals = final_data.groupby(['State', 'City'], observed=True).size().nlargest(30)
city_totals.index = [f'{city}, {state}' for state, city in city_totals.index]
city_totals.sort_values().plot.barh(figsize=(12, 10))
plt.title('30 lokalizacji z największą liczbą zarejestrowanych zdarzeń')
plt.xlabel('Liczba rekordów')
plt.ylabel('Miasto, stan')
plt.tight_layout()
plt.show()


**Obserwacje dotyczące lokalizacji.** Po czyszczeniu pozostało 49 różnych kodów `State` i 12 237 różnych nazw miast. Nazwa miasta nie jest jednoznacznym identyfikatorem lokalizacji, dlatego ranking uwzględnia również stan. Wśród pokazanych lokalizacji najwięcej rekordów ma Miami, następnie Houston i Los Angeles. Są to liczby zarejestrowanych zdarzeń, a nie wskaźniki ryzyka: nie uwzględniają liczby mieszkańców, natężenia ruchu ani kompletności rejestracji.

# Przygotowanie znaczników czasu

In [ ]:
final_data.Start_Time = pd.to_datetime(final_data.Start_Time, format='mixed')
final_data.Start_Time[0]

## Podzbiory według czasu

In [ ]:
final_data['Month'] = final_data['Start_Time'].dt.month
final_data['Year'] = final_data['Start_Time'].dt.year
final_data['Hour'] = final_data['Start_Time'].dt.hour
final_data['Weekday'] = final_data['Start_Time'].dt.day_of_week
#yearly data subset 
data_2016 = final_data[final_data.Start_Time.dt.year == 2016]
data_2017 = final_data[final_data.Start_Time.dt.year == 2017]
data_2018 = final_data[final_data.Start_Time.dt.year == 2018]
data_2019 = final_data[final_data.Start_Time.dt.year == 2019]
data_2020 = final_data[final_data.Start_Time.dt.year == 2020]
data_2021 = final_data[final_data.Start_Time.dt.year == 2021]
data_2022 = final_data[final_data.Start_Time.dt.year == 2022]
data_2023 = final_data[final_data.Start_Time.dt.year == 2023]
data_2017_2019 = final_data[(final_data["Year"] >= 2017) & (final_data["Year"] <= 2019)]

# Roczna analiza wypadków

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Year", data=final_data, orient = 'v', palette = "crest")
plt.annotate('Brak danych',xy=(-0.4,500000), fontsize=11)
c.set_title("Liczba wypadków według roku")
for i in ax.patches:
    count = '{:,.0f}'.format(i.get_height())
    x = i.get_x()+i.get_width()-0.60
    y = i.get_height()+10000
    ax.annotate(count, (x, y))
plt.show()

# Miesięczna analiza wypadków

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Month", data=data_2016, orient = 'v', palette = "crest")
plt.annotate('Brak danych',xy=(1,10000), fontsize=11)
c.set_title("Liczba wypadków według miesiąca w 2016 roku")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Month", data=data_2017_2019, orient = 'v', palette = "crest")
c.set_title("Liczba wypadków według miesiąca w latach 2017-2019")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Month", data=data_2020, orient = 'v', palette = "crest")
plt.annotate('Pandemia COVID-19',xy=(2,120000), fontsize=12)
plt.annotate("[",xy=(0,0),xytext=(1.9,120000),arrowprops={'arrowstyle':'-|>'}, fontsize=12)
plt.annotate("]",xy=(10,0),xytext=(4.5,120000),arrowprops={'arrowstyle':'-|>'}, fontsize=12)
c.set_title("Liczba wypadków według miesiąca w 2020 roku")
plt.show()

# Tygodniowa analiza wypadków

## Wszystkie tygodnie 2022 roku

In [ ]:
fig, ax = plt.subplots(figsize = (15,5))
c = sns.countplot(x=data_2022.Start_Time.dt.isocalendar().week, data=data_2022, orient = 'v', palette = "crest")
plt.annotate('Brak danych',xy=(3,2500), fontsize=11)
c.set_title("Liczba rekordów według tygodni ISO w 2022 roku")
plt.show()

## Wszystkie tygodnie w latach 2017-2019

In [ ]:
fig, ax = plt.subplots(figsize = (15,5))
c = sns.countplot(x=data_2017_2019.Start_Time.dt.isocalendar().week, data=data_2017_2019, orient = 'v', palette = "crest")
c.set_title("Liczba rekordów według tygodni ISO w latach 2017–2019")
plt.show()

## Dni tygodnia

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Weekday", data=final_data, orient = 'v', palette = "crest_r")
plt.annotate('Dni robocze',xy=(2,150000), fontsize=12)
plt.annotate('Weekend',xy=(5.1,150000), fontsize=12)
c.set_title("Liczba wypadków według dnia tygodnia")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
sns.countplot(x="Hour", data=final_data, orient = 'v', palette = "icefire_r")
plt.annotate('Poranny szczyt',xy=(6,105000), fontsize=12)
plt.annotate('Wieczorny szczyt',xy=(15,105000), fontsize=12)
plt.annotate('dojazd do pracy',xy=(9,0),xytext=(3,95000),arrowprops={'arrowstyle':'-|>'}, fontsize=12)
plt.annotate('powrót z pracy',xy=(17,0),xytext=(19,95000),arrowprops={'arrowstyle':'-|>'}, fontsize=12)
plt.show()

## Analiza wypadków według stopnia ciężkości

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
c = sns.countplot(x="Severity", data=final_data, orient = 'v', palette = "crest_r")
c.set_title("Stopień ciężkości wypadków")
for i in ax.patches:
    count = "{:.2%}".format(i.get_height()/len(final_data.Severity))
    x = i.get_x()+i.get_width()-0.50
    y = i.get_height()+20000
    ax.annotate(count, (x, y))
plt.show()

**Wniosek o klasach.** W danych źródłowych klasa 2 obejmuje 79,67% rekordów, klasa 3 — 16,81%, klasa 4 — 2,65%, a klasa 1 — 0,87%. Jest to silnie niezbalansowany problem klasyfikacji. Samo przewidywanie klasy 2 dawałoby około 79,67% accuracy na zbiorze o tych proporcjach, dlatego taki wynik nie wystarcza do oceny użyteczności modelu. Po czyszczeniu dominacja klasy 2 pozostaje praktycznie taka sama.

# Analiza danych pogodowych i lokalizacji wypadków

In [ ]:
state_counts = pd.crosstab(final_data['State'], final_data['Severity'])
state_top = state_counts.sum(axis=1).nlargest(20).index
state_percent = state_counts.loc[state_top].div(state_counts.loc[state_top].sum(axis=1), axis=0) * 100
state_percent.index = [f"{state} (n={state_counts.loc[state].sum():,})" for state in state_top]
state_percent.plot.bar(stacked=True, figsize=(15, 6), colormap='viridis')
plt.title('Udział klas Severity — 20 stanów z największą liczbą rekordów')
plt.ylabel('Udział w danym stanie [%]')
plt.xlabel('Stan i liczba rekordów')
plt.legend(title='Severity')
plt.tight_layout()
plt.show()

In [ ]:
city_counts = final_data.groupby(['State', 'City', 'Severity'], observed=True).size().unstack('Severity', fill_value=0)
city_top = city_counts.sum(axis=1).nlargest(20).index
city_percent = city_counts.loc[city_top].div(city_counts.loc[city_top].sum(axis=1), axis=0) * 100
city_percent.index = [f"{city}, {state} (n={city_counts.loc[(state, city)].sum():,})" for state, city in city_top]
city_percent.plot.barh(stacked=True, figsize=(12, 9), colormap='viridis')
plt.title('Udział klas Severity — 20 najliczniejszych lokalizacji')
plt.xlabel('Udział w danej lokalizacji [%]')
plt.ylabel('Miasto, stan i liczba rekordów')
plt.legend(title='Severity')
plt.tight_layout()
plt.show()

**Obserwacje dotyczące struktury klas w lokalizacjach.** Klasa 2 dominuje w pokazanych stanach, lecz udziały klas istotnie się różnią. Georgia i Illinois mają większy udział klasy 3 niż np. Floryda czy Karolina Południowa. Wśród miast Atlanta wyróżnia się wysokim udziałem klasy 3, a Tucson — klasy 1. Lokalizacja może zatem zawierać informację przydatną do klasyfikacji; różnice mogą też odzwierciedlać źródła i sposób rejestracji danych.

# Warunki pogodowe a klasy Severity

#### Powiązane kolumny

In [ ]:
weather_columns = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
                   'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition']
# Do analizy rozkładów używamy pomiarów źródłowych, bez imputacji średnią.
weather = df.loc[:, weather_columns + ['Severity']].copy()
weather.head()

## Wartości odstające — porównanie przed i po filtrze IQR

In [ ]:
box_sample = weather.sample(n=min(10_000, len(weather)), random_state=42)
box_columns = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)']
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, column in zip(axes.flat, box_columns):
    sns.boxplot(data=box_sample, x='Severity', y=column, ax=ax, color='skyblue')
    ax.set_title(column)
fig.suptitle('Pomiary według klas — przed filtrem IQR (próbka losowa)')
plt.tight_layout()
plt.show()

In [ ]:
def remove_outliers(data, name):
    q25, q75 = data[name].quantile([0.25, 0.75])
    margin = 1.5 * (q75 - q25)
    # Zachowujemy braki: raport dotyczy odrzucania zmierzonych wartości.
    return data.loc[data[name].isna() | data[name].between(q25 - margin, q75 + margin)]

In [ ]:
# IQR jest wariantem porównawczym, a nie automatycznym czyszczeniem danych.
weather_iqr = weather
for column in ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)']:
    weather_iqr = remove_outliers(weather_iqr, name=column)
iqr_audit = pd.DataFrame({
    'przed': weather['Severity'].value_counts(),
    'po_IQR': weather_iqr['Severity'].value_counts(),
}).fillna(0).astype(int).sort_index()
iqr_audit['usunięto_%'] = (1 - iqr_audit['po_IQR'] / iqr_audit['przed']) * 100
display(iqr_audit)
# Kolejne analizy nadal korzystają z weather, bez filtrowania IQR.

In [ ]:
box_sample = weather_iqr.sample(n=min(10_000, len(weather_iqr)), random_state=42)
box_columns = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)']
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, column in zip(axes.flat, box_columns):
    sns.boxplot(data=box_sample, x='Severity', y=column, ax=ax, color='skyblue')
    ax.set_title(column)
fig.suptitle('Pomiary według klas — po filtrze IQR (próbka losowa)')
plt.tight_layout()
plt.show()

**Wpływ filtra IQR.** Sekwencyjne filtrowanie pięciu pomiarów usuwa 13,12% rekordów klasy 1, 9,04% klasy 2, 6,26% klasy 3 i 10,76% klasy 4. Filtr nie działa więc jednakowo na wszystkie klasy i zmienia skład analizowanego zbioru. Ponieważ ekstremalne warunki mogą być rzeczywiste, wariant po IQR służy porównaniu; dalsze rozkłady pogody korzystają z danych przed tym filtrem. Pokazane boxploty opierają się na próbkach losowych, a nie na wszystkich rekordach.

## Temperatura (°F)

In [ ]:
print("Unikalne wartości: ",weather["Temperature(F)"].nunique())
print("Brakujące wartości: ",weather["Temperature(F)"].isna().sum())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Temperature(F)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Temperature(F)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Temperature(F) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Temperatura odczuwalna (°F)

In [ ]:
print("Unikalne wartości: ",weather["Wind_Chill(F)"].nunique())
print("Brakujące wartości: ",weather["Wind_Chill(F)"].isna().sum())
print("Średnia: ",weather["Wind_Chill(F)"].mean())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Wind_Chill(F)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Wind_Chill(F)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Wind_Chill(F) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Wilgotność (%)

In [ ]:
print("Unikalne wartości: ",weather["Humidity(%)"].nunique())
print("Brakujące wartości: ",weather["Humidity(%)"].isna().sum())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Humidity(%)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Humidity(%)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Humidity(%) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Ciśnienie (cale słupa rtęci)

In [ ]:
print("Brakujące wartości: ",weather["Pressure(in)"].isna().sum())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Pressure(in)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Pressure(in)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Pressure(in) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Widoczność (mile)

In [ ]:
print("Unikalne wartości: ",weather["Visibility(mi)"].nunique())
print("Brakujące wartości: ",weather["Visibility(mi)"].isna().sum())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Visibility(mi)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Visibility(mi)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Visibility(mi) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Kierunek wiatru

In [ ]:
print("Unikalne kategorie: ",weather["Wind_Direction"].unique())
print("Unikalne wartości: ",weather["Wind_Direction"].nunique())

In [ ]:
f,ax=plt.subplots(2,figsize=(16,10))
sns.countplot(x="Wind_Direction",
              data=weather,
              order=weather.Wind_Direction.value_counts().iloc[0:12].index,
              palette="Set2",
              ax=ax[0])
sns.countplot(x="Wind_Direction",
              data=weather,
              order=weather.Wind_Direction.value_counts().iloc[12:24].index,
              palette="Set2",
              ax=ax[1])

## Prędkość wiatru (mile/h)

In [ ]:
print("Unikalne wartości: ",weather["Wind_Speed(mph)"].nunique())

In [ ]:
# Ograniczenie liczby obserwacji w każdej klasie chroni pamięć i rzadkie klasy.
kde_sample = pd.concat([
    group.sample(n=min(2000, len(group)), random_state=42)
    for _, group in weather[['Severity', 'Wind_Speed(mph)']].groupby('Severity', observed=True)
])
sns.displot(data=kde_sample, x='Wind_Speed(mph)', hue='Severity', kind='kde',
            common_norm=False, cut=0, palette='Set2', height=4, aspect=2)
plt.title('Rozkład Wind_Speed(mph) w klasach — normalizacja osobno dla każdej klasy')
plt.show()

## Warunki pogodowe

In [ ]:
f,ax=plt.subplots(5,figsize=(25,16))
sns.countplot(x="Weather_Condition",
              data=weather,
              order=weather.Weather_Condition.value_counts().iloc[0:10].index,
              palette="Set2",
              ax=ax[0])
sns.countplot(x="Weather_Condition",
              data=weather,
              order=weather.Weather_Condition.value_counts().iloc[10:20].index,
              palette="Set2",
              ax=ax[1])
sns.countplot(x="Weather_Condition",
              data=weather,
              order=weather.Weather_Condition.value_counts().iloc[20:30].index,
              palette="Set2",
              ax=ax[2])
sns.countplot(x="Weather_Condition",
              data=weather,
              order=weather.Weather_Condition.value_counts().iloc[30:40].index,
              palette="Set2",
              ax=ax[3])
sns.countplot(x="Weather_Condition",
              data=weather,
              order=weather.Weather_Condition.value_counts().iloc[40:50].index,
              palette="Set2",
              ax=ax[4])

# Infrastruktura a klasy Severity

#### Powiązane kolumny

In [ ]:
location = final_data.iloc[:, 26:39]
location.head()

In [ ]:
infrastructure_columns = list(location.columns)
fig, axes = plt.subplots(5, 3, figsize=(17, 20))
for ax, column in zip(axes.flat, infrastructure_columns):
    counts = pd.crosstab(final_data[column], final_data['Severity'])
    percentages = counts.div(counts.sum(axis=1), axis=0) * 100
    percentages.index = [f'{value} (n={counts.loc[value].sum():,})' for value in percentages.index]
    percentages.plot.bar(stacked=True, ax=ax, legend=False, colormap='viridis', rot=0)
    ax.set(title=column, xlabel='', ylabel='Udział klas [%]', ylim=(0, 100))
for ax in axes.flat[len(infrastructure_columns):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Severity', loc='upper right')
plt.tight_layout()
plt.show()

**Obserwacje dotyczące infrastruktury.** Przy `Crossing=True` i `Traffic_Signal=True` udział klasy 2 jest większy niż przy wartości `False`. Dla `Junction=True` widoczny jest natomiast większy udział klas 3 i 4. `Roundabout=True` występuje po czyszczeniu jedynie w 243 rekordach, więc porównanie tej grupy jest znacznie mniej stabilne. Wykresy opisują strukturę klas wśród zdarzeń, a nie wpływ infrastruktury na prawdopodobieństwo wystąpienia wypadku.

## Uzupełnienie EDA: jakość danych i zależności względem Severity

In [ ]:
# Pełne dane do tabel; ograniczone próbki tylko do kosztownych wizualizacji.
category_columns = df.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.difference(['ID', 'Description'])
category_rows = []
for column in category_columns:
    counts = df[column].value_counts(dropna=True)
    category_rows.append({
        'cecha': column, 'unikalne': df[column].nunique(),
        'najczęstsza': counts.index[0] if len(counts) else None,
        'udział_najczęstszej_%': 100 * counts.iloc[0] / counts.sum() if len(counts) else np.nan,
        'braki_%': 100 * df[column].isna().mean(),
    })
category_summary = pd.DataFrame(category_rows).set_index('cecha')
display(category_summary.sort_values('unikalne'))
print('Powtórzone identyfikatory ID:', df['ID'].duplicated().sum())
print('Cechy stałe:', df.columns[df.nunique(dropna=False) <= 1].tolist())
# Pokazujemy równoważne zapisy kierunków bez modyfikowania df.
wind_aliases = {'CALM': 'Calm', 'VAR': 'Variable', 'East': 'E', 'West': 'W', 'North': 'N', 'South': 'S'}
display(pd.DataFrame({'oryginał': df['Wind_Direction'].dropna().unique()}).assign(
    ujednolicona=lambda table: table['oryginał'].astype(str).replace(wind_aliases)))

**Wnioski o jakości cech.** `Country` i `Turning_Loop` są stałe w całym zbiorze, więc nie rozróżniają obserwacji. `Roundabout`, `Bump` i `Traffic_Calming` są bardzo rzadkie. `Wind_Direction` zawiera równoważne zapisy, m.in. `Calm`/`CALM`, `Variable`/`VAR` oraz `West`/`W`; tabela pokazuje proponowane ujednolicenie, ale nie zmienia danych. Brak powtórzonych `ID` potwierdza unikalność identyfikatorów, lecz nie wyklucza wielokrotnego opisania tego samego zdarzenia pod różnymi identyfikatorami.

### Wpływ czyszczenia na liczebność klas

In [ ]:
cleaning_audit = pd.DataFrame({
    'źródło_n': df['Severity'].value_counts(),
    'po_czyszczeniu_n': final_data['Severity'].value_counts(),
}).fillna(0).sort_index()
cleaning_audit['źródło_%'] = cleaning_audit['źródło_n'] / len(df) * 100
cleaning_audit['po_czyszczeniu_%'] = cleaning_audit['po_czyszczeniu_n'] / len(final_data) * 100
cleaning_audit['usunięto_w_klasie_%'] = (1 - cleaning_audit['po_czyszczeniu_n'] / cleaning_audit['źródło_n']) * 100
display(cleaning_audit.round(3))
print(f'Usunięte rekordy: {len(df) - len(final_data):,} ({100 * (1 - len(final_data) / len(df)):.2f}%)')
# Braki według klasy pozwalają sprawdzić, czy czyszczenie nie działa wybiórczo.
missing_by_class = df.groupby('Severity', observed=True)[weather_columns].agg(lambda values: values.isna().mean() * 100)
display(missing_by_class.round(2))

**Wpływ czyszczenia i braków na klasy.** Usunięto 291 451 rekordów (3,77%), pozostawiając 7 436 943 obserwacje. Największą względną stratę ma klasa 4 (6,13%); jej udział maleje z 2,649% do 2,584%. Globalne proporcje zmieniają się niewiele, ale usuwanie danych nie jest jednakowe dla klas. Szczególnie wyraźne różnice dotyczą braków temperatury odczuwalnej (3,28% w klasie 1 wobec 46,10% w klasie 3) i opadów (5,81% wobec 49,29%). Oznacza to, że interpretując pomiary trzeba uwzględnić ich różną dostępność, bez zakładania, że braki są losowe.

### Statystyki pomiarów w klasach — bez imputacji i filtra IQR

In [ ]:
numeric_eda = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
               'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Distance(mi)']
display(df.groupby('Severity', observed=True)[numeric_eda].agg(['count', 'median', 'mean', 'std']).round(3))
display(df.groupby('Severity', observed=True)[numeric_eda].quantile([0.25, 0.75]).round(3))
# Próbka do wykresów: do 2000 rekordów z każdej klasy, bez zmiany danych źródłowych.
eda_sample = pd.concat([
    group[numeric_eda + ['Severity']].sample(n=min(2000, len(group)), random_state=42)
    for _, group in df[numeric_eda + ['Severity']].groupby('Severity', observed=True)
])

**Porównanie pomiarów w klasach.** Mediana temperatury wynosi 73°F w klasie 1, 64°F w klasie 2, około 64,9°F w klasie 3 i 60,1°F w klasie 4. Różnice występują, lecz zakresy międzykwartylowe mocno się nakładają. Dla `Distance(mi)` klasa 4 ma medianę 0,473 mili i średnią 1,496 mili, wyższe niż pozostałe klasy. Mediana opadów wynosi zero we wszystkich klasach, a kwartyle widoczności 25% i 75% wynoszą wszędzie 10 mil — same te statystyki nie rozróżniają klas. Różnice nie dowodzą zależności przyczynowej ani przydatności predykcyjnej poza tym zbiorem.

### Histogramy: skala oryginalna i log1p (tylko wartości nieujemne)

In [ ]:
hist_columns = ['Distance(mi)', 'Precipitation(in)', 'Visibility(mi)', 'Wind_Speed(mph)']
hist_sample = df[hist_columns].sample(n=min(50_000, len(df)), random_state=42)
fig, axes = plt.subplots(len(hist_columns), 2, figsize=(13, 13))
for row, column in enumerate(hist_columns):
    values = hist_sample[column].dropna()
    sns.histplot(values, bins=40, ax=axes[row, 0])
    axes[row, 0].set_title(column)
    sns.histplot(np.log1p(values[values >= 0]), bins=40, ax=axes[row, 1])
    axes[row, 1].set_title(f'log1p({column}); wykluczono ujemne: {(values < 0).sum()}')
plt.tight_layout()
plt.show()

**Wnioski o rozkładach.** Statystyki pełnego zbioru potwierdzają silną skośność opadów i `Distance(mi)`: średnie przekraczają mediany, a nieliczne duże wartości rozciągają skale. Dla opadów oba kwartyle 25% i 75% wynoszą zero, dlatego warto odróżniać samo wystąpienie opadu od jego wielkości. Skala `log1p` ułatwia oglądanie długiego prawego ogona, ale nie usuwa zer ani nie weryfikuje wiarygodności ekstremów. Histogramy przedstawiają próbkę do 50 000 rekordów.

### Zależności między pomiarami — korelacja Spearmana

In [ ]:
# Losowa próbka zachowuje naturalne proporcje klas; Severity nie jest pomiarem ciągłym.
corr_sample = df[numeric_eda].sample(n=min(50_000, len(df)), random_state=42)
correlation = corr_sample.corr(method='spearman')
plt.figure(figsize=(11, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Korelacja rang Spearmana — próbka do 50 000 rekordów')
plt.tight_layout()
plt.show()

**Wnioski z korelacji.** W losowej próbce korelacja Spearmana temperatury i temperatury odczuwalnej zaokrągla się do 1,00 — te cechy przekazują bardzo podobną informację w dostępnych parach pomiarów. Wilgotność jest ujemnie związana z widocznością (około −0,47), a opady z widocznością (około −0,44); opady i wilgotność mają korelację około 0,37. `Distance(mi)` ma słabe korelacje z pokazanymi pomiarami pogody. Korelacje obliczane są na dostępnych parach wartości i nie przesądzają o związku z `Severity`.

### Pogoda i pora dnia: procentowe rozkłady klas

In [ ]:
for column in ['Sunrise_Sunset', 'Weather_Condition']:
    counts = pd.crosstab(df[column], df['Severity'])
    top = counts.sum(axis=1).nlargest(12).index
    percentages = counts.loc[top].div(counts.loc[top].sum(axis=1), axis=0) * 100
    percentages.index = [f'{value} (n={counts.loc[value].sum():,})' for value in top]
    percentages.plot.barh(stacked=True, figsize=(12, 6), colormap='viridis')
    plt.title(f'Rozkład Severity według {column} — dane źródłowe')
    plt.xlabel('Udział klas w grupie [%]')
    plt.legend(title='Severity')
    plt.tight_layout()
    plt.show()

**Obserwacje dotyczące pogody i pory dnia.** W obu porach doby dominuje klasa 2. Dla nocy widoczny jest większy udział klasy 4, mimo mniejszej łącznej liczby rekordów niż za dnia. Wśród pokazanych kategorii pogodowych `Clear`, `Overcast` i `Scattered Clouds` mają większy udział klasy 3 niż `Fair`. Nie ma prostego wzorca, w którym każda kategoria niekorzystnej pogody oznacza większy udział wyższych klas. Różnice mogą być powiązane także z lokalizacją, okresem i źródłem danych; sam wykres nie pozwala ich rozdzielić.

### Pokrycie czasu oraz średnia liczba rekordów na obserwowany dzień

In [ ]:
daily_counts = final_data.groupby(final_data['Start_Time'].dt.floor('D')).size()
calendar = pd.date_range(daily_counts.index.min(), daily_counts.index.max(), freq='D')
coverage = daily_counts.reindex(calendar)
print('Zakres dat:', calendar.min(), '—', calendar.max())
print('Dni bez rekordów w tym zakresie:', coverage.isna().sum())
annual_coverage = coverage.groupby(coverage.index.year).agg(['count', 'size', 'sum', 'mean'])
annual_coverage.columns = ['dni_z_rekordami', 'dni_w_zakresie', 'rekordy', 'średnia_na_dzień_z_rekordami']
display(annual_coverage)
annual_coverage['średnia_na_dzień_z_rekordami'].plot.bar(figsize=(10, 4))
plt.ylabel('Rekordy / dzień z rekordami')
plt.title('Porównanie lat; dni bez rekordów nie są traktowane jako zero wypadków')
plt.tight_layout()
plt.show()
hour_weekday = pd.crosstab(final_data['Weekday'], final_data['Hour']).reindex(index=range(7), columns=range(24), fill_value=0)
weekday_days = daily_counts.groupby(daily_counts.index.day_of_week).size().reindex(range(7), fill_value=0)
plt.figure(figsize=(14, 5))
sns.heatmap(hour_weekday.div(weekday_days.replace(0, np.nan), axis=0), cmap='viridis',
            yticklabels=['pon', 'wt', 'śr', 'czw', 'pt', 'sob', 'nd'])
plt.title('Średnia liczba rekordów w godzinie na obserwowany dzień danego typu')
plt.xlabel('Godzina')
plt.ylabel('Dzień tygodnia')
plt.tight_layout()
plt.show()
for column in ['Hour', 'Weekday']:
    counts = pd.crosstab(final_data[column], final_data['Severity'])
    display(counts.assign(liczebność=counts.sum(axis=1)))
    (counts.div(counts.sum(axis=1), axis=0) * 100).plot.bar(stacked=True, figsize=(12, 4), colormap='viridis')
    plt.title(f'Procentowy rozkład Severity według {column}')
    plt.ylabel('Udział klas [%]')
    plt.tight_layout()
    plt.show()

**Wnioski dotyczące czasu.** Po czyszczeniu zakres obejmuje 14.01.2016–31.03.2023; wewnątrz niego są 62 dni bez rekordów. Rok 2023 obejmuje tylko początek roku, a liczba dni z rekordami różni się między latami. Średnia liczba rekordów na dzień z rekordami rośnie od około 1225 w 2016 r. do 4804 w 2022 r.; nie dowodzi to wzrostu ryzyka, ponieważ mogła zmieniać się kompletność rejestracji.

Najwięcej rekordów przypada na godziny 7–8 i 16–17. Piątek ma najwięcej obserwacji wśród dni tygodnia, a niedziela najmniej. Jednocześnie udział klasy 4 o północy wynosi około 6,31%, wobec 1,66% o godzinie 7; w niedzielę to około 3,86%, wobec 2,35% w piątek. Największa liczba zdarzeń nie oznacza więc największego udziału tej klasy. Adnotacje o dojazdach do pracy lub pandemii we wcześniejszych wykresach należy traktować jako hipotezy interpretacyjne, a nie potwierdzone przyczyny.

### Relacje wielowymiarowe — próbka z każdej klasy

In [ ]:
pair_columns = ['Temperature(F)', 'Humidity(%)', 'Wind_Speed(mph)']
pair_sample = pd.concat([
    group.sample(n=min(400, len(group)), random_state=42)
    for _, group in eda_sample.groupby('Severity', observed=True)
])[pair_columns + ['Severity']].dropna()
pair_sample['Severity'] = pair_sample['Severity'].astype(str)
if len(pair_sample) > 1:
    sns.pairplot(pair_sample, vars=pair_columns, hue='Severity', diag_kind='hist',
                 plot_kws={'s': 10, 'alpha': 0.3}, height=2.4)
    plt.show()

**Obserwacje wielowymiarowe.** Na próbce temperatury, wilgotności i prędkości wiatru klasy silnie się nakładają; nie widać prostego rozdzielenia ich pojedynczym progiem ani wyraźnie odseparowanych skupień. Klasa 1 jest bardziej skupiona przy wyższych temperaturach, zgodnie z medianami z pełnego zbioru. Widoczne pionowe i poziome pasma wskazują na powtarzalne, dyskretne wartości pomiarów. Wykres służy rozpoznaniu zależności; nie zastępuje oceny modelu.

Próbki z równym limitem na klasę służą porównaniu kształtów rozkładów, a nie oszacowaniu udziałów klas. Tabele liczebności i wykresy procentowe korzystają z pełnych danych. Zależności dotyczą zarejestrowanych zdarzeń; bez danych o natężeniu ruchu nie określają ryzyka wypadku ani związku przyczynowego.

### Podsumowanie wniosków z EDA

- Zbiór jest silnie niezbalansowany; dominacja klasy 2 wymaga oceny jakości osobno dla klas.
- Czyszczenie zmienia globalne proporcje nieznacznie, ale częściej usuwa rekordy klasy 4. Dostępność pomiarów pogody silnie zależy od klasy.
- Rozkłady klas różnią się między lokalizacjami, porami doby i grupami infrastruktury. Te zależności są kandydatami do weryfikacji podczas modelowania.
- Temperatura i temperatura odczuwalna niosą podobną informację; `Country` i `Turning_Loop` są stałe. Kategorie kierunku wiatru wymagają ujednolicenia.
- Skośność rozkładów oraz nierówny wpływ filtra IQR przemawiają za ostrożnym traktowaniem ekstremów. Zera, braki i rzeczywiste rzadkie pomiary mają różne znaczenie.
- EDA opisuje zarejestrowane zdarzenia. Bez informacji o ekspozycji na ruch i sposobie zbierania danych nie wyznacza ryzyka wypadku ani przyczyn obserwowanych różnic.

# Przygotowanie danych do modelowania stopnia ciężkości wypadku

Ta sekcja wykonuje tylko preprocessing: nie trenuje ani nie ocenia modelu. Oryginalna ramka `df` i jej kolumny pozostają bez zmian. Wynikami są przygotowane cechy, wagi klas i indeksy foldów. Przy późniejszej ocenie modelu preprocessing należy dopasowywać osobno w każdym foldzie, aby uniknąć przecieku danych.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

TARGET_COLUMN = "Severity"
CV_FOLDS = 5
CV_SAMPLE_LIMIT = 100_000
RANDOM_STATE = 42

if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Missing target column: {TARGET_COLUMN}")
if "Start_Time" not in df.columns:
    raise KeyError("Missing Start_Time required for time feature engineering")

numeric_candidates = [
    "Start_Lat", "Start_Lng", "Distance(mi)", "Temperature(F)",
    "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)",
]
categorical_candidates = [
    "Source", "State", "Weather_Condition", "Wind_Direction", "Timezone",
    "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight",
    "Astronomical_Twilight",
]
binary_candidates = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop",
]

numeric_features = [column for column in numeric_candidates if column in df.columns]
categorical_features = [column for column in categorical_candidates if column in df.columns]
categorical_features += [column for column in binary_candidates if column in df.columns]
model_feature_columns = numeric_features + categorical_features
if not model_feature_columns:
    raise ValueError("No model input features were found")

y_all = df.loc[df[TARGET_COLUMN].notna(), TARGET_COLUMN]
class_counts = y_all.value_counts()
if len(class_counts) < 2:
    raise ValueError("At least two Severity classes are required")
if class_counts.min() < CV_FOLDS:
    raise ValueError(
        f"The rarest class has {class_counts.min()} records; "
        f"not enough for {CV_FOLDS}-fold cross-validation"
    )

if len(y_all) > CV_SAMPLE_LIMIT:
    model_indices, _ = train_test_split(
        y_all.index.to_numpy(),
        train_size=CV_SAMPLE_LIMIT,
        random_state=RANDOM_STATE,
        stratify=y_all.to_numpy(),
    )
    model_indices = pd.Index(model_indices)
else:
    model_indices = y_all.index

# Work on a copy so the source dataframe and its columns remain unchanged.
X_model = df.loc[model_indices, model_feature_columns].copy()
y_model = y_all.loc[model_indices].copy()

start_time = pd.to_datetime(
    df.loc[model_indices, "Start_Time"], errors="coerce", format="mixed"
)
engineered_features = {
    "Hour_sin": np.sin(2 * np.pi * start_time.dt.hour / 24),
    "Hour_cos": np.cos(2 * np.pi * start_time.dt.hour / 24),
    "Weekday_sin": np.sin(2 * np.pi * start_time.dt.day_of_week / 7),
    "Weekday_cos": np.cos(2 * np.pi * start_time.dt.day_of_week / 7),
    "Month_sin": np.sin(2 * np.pi * (start_time.dt.month - 1) / 12),
    "Month_cos": np.cos(2 * np.pi * (start_time.dt.month - 1) / 12),
}
if set(engineered_features).intersection(df.columns):
    raise ValueError("Engineered feature names conflict with existing df columns")
for feature_name, values in engineered_features.items():
    X_model[feature_name] = values
numeric_features += list(engineered_features)

for column in numeric_features:
    X_model[column] = pd.to_numeric(X_model[column], errors="coerce")
for column in categorical_features:
    X_model[column] = X_model[column].astype("object").where(
        X_model[column].notna(), np.nan
    )

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Prepare class weights and CV folds; no classifier is trained here.
sample_weights = pd.Series(
    compute_sample_weight(class_weight="balanced", y=y_model),
    index=y_model.index,
    name="sample_weight",
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv.split(X_model, y_model))

# Fit only the preprocessing transformations; no estimator is trained.
X_preprocessed = preprocessor.fit_transform(X_model)
prepared_feature_names = preprocessor.get_feature_names_out()
print("Class counts in source data:\n", class_counts)
print(f"Records in preprocessing sample: {len(y_model):,}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical/binary features: {len(categorical_features)}")
print(f"Prepared feature matrix shape: {X_preprocessed.shape}")
print(f"Prepared feature names: {len(prepared_feature_names)}")
print(f"Stratified CV folds: {len(cv_splits)}")
print("Sample weights:", sample_weights.describe())